# Generate the LCA impact scores

In [1]:
# Import the required libraries
from mescal import *
import pandas as pd
import bw2data as bd

## Set up and input data

In [2]:
ecoinvent_version = '3.12' # choose the ecoinvent version you wish to use
regionalize_foregrounds = ['Operation', 'Resource'] # set to 'all' if you want to regionalize the foreground inventories of all types of LCI datasets
premise_iam = 'image' # choose the IAM to which the premise database is linked
premise_ssp_rcp = 'SSP2-M' # choose the SSP/RCP scenario to which the premise database is linked
years = [2050] # choose the years for which you want to generate metrics

In [3]:
# Set the name of the new database with the ESM datasets (the different years will be added to that name)
esm_db_name = 'PyPSA-Eur'

In [4]:
# Main version of ecoinvent (without .1 if any)
ecoinvent_main_version = '.'.join(ecoinvent_version.split('.')[:2])

In [5]:
lcia_methods=['EF v3.1']

In [6]:
# Set up your Brightway project
bd.projects.set_current('pypsa') # put the name of your brightway project here

In [7]:
# Input data files for all time steps
input_data_folder = 'your_specific_input_files'
mapping = pd.read_csv(f'{input_data_folder}/mapping.csv')
unit_conversion = pd.read_csv(f'{input_data_folder}/unit_conversion.csv')
mapping_esm_flows_to_CPC = pd.read_csv(f'{input_data_folder}/mapping_esm_flows_to_CPC.csv')
technology_compositions = pd.read_csv(f'{input_data_folder}/technology_compositions.csv')
technology_specifics = pd.read_csv(f'{input_data_folder}/technology_specifics.csv')
efficiency = pd.read_csv(f'{input_data_folder}/efficiency.csv')
impact_abbrev = pd.read_csv('generic_input_files/impact_abbrev.csv')
mapping_new_products_to_cpc = pd.read_csv('generic_input_files/mapping_new_products_to_CPC.csv')

In [8]:
def create_custom_dataset_for_process_emissions(db_name):
    code = random_code()

    return {
        'name': 'carbon dioxide emissions from processes',
        'reference product': 'carbon dioxide, emitted',
        'unit': 'kilogram',
        'location': 'GLO',
        'exchanges': [
            {
                'name': 'carbon  dioxide, emitted',
                'amount': 1.0,
                'unit': 'kilogram',
                'type': 'production',
                'location': 'GLO',
                'code': code,
                'database': db_name,
            },
            {
                'input': ('biosphere3', '349b29d1-3e58-4c66-98b9-9d1a076efd2e'),
                'name': 'Carbon dioxide, fossil',
                'amount': 1.0,
                'unit': 'kilogram',
                'database': 'biosphere3',
                'code': '349b29d1-3e58-4c66-98b9-9d1a076efd2e',
                'type': 'biosphere',
                'categories': ('air',),
            }
        ],
        'comment': 'Custom dataset for the CO2 emissions of processes in PyPSA',
        'database': db_name,
        'code': code,
        'classifications': [('CPC', '0000: Custom CPC')],
    }

In [9]:
# Time-dependent input data files
time_steps = []
for index, year in enumerate(years):
    model = pd.read_csv(f'{input_data_folder}/{year}/model.csv')
    lifetime = pd.read_csv(f'{input_data_folder}/{year}/lifetime.csv')
    main_database_name = f"ecoinvent_cutoff_{ecoinvent_version}_{premise_iam}_{premise_ssp_rcp}_{year}"
    main_database = Database(main_database_name, create_pickle=True)
    process_emissions_ds = create_custom_dataset_for_process_emissions(main_database_name)
    main_database += Database(db_as_list=[process_emissions_ds])
    main_database.add_CPC_categories(
        overwrite_existing_CPC=True, 
        mapping_new_products_to_CPC=mapping_new_products_to_cpc,
        )  # the CPC categories of CO2 flows need to be overwritten (same category as H2)
    time_steps.append({'year': year, 'model': model, 'lifetime': lifetime, 'main_database': main_database, 'main_database_name': main_database_name})

    # The database name in the mapping file should be the one of the first time step
    if index == 0:
        mapping.Database = main_database_name

2026-11-06 17:53:13,425 - Database - INFO - Loaded ecoinvent_cutoff_3.12_image_SSP2-M_2050 from pickle!


## Initialize the PathwayESM class

In [10]:
esm = PathwayESM(
    time_steps=time_steps,
    mapping=mapping,
    unit_conversion=unit_conversion,
    mapping_esm_flows_to_CPC_cat=mapping_esm_flows_to_CPC,
    esm_db_name=esm_db_name,
    biosphere_db_name='biosphere3',
    esm_location='RER',
    technology_compositions=technology_compositions,
    tech_specifics=technology_specifics,
    efficiency=efficiency,
    regionalize_foregrounds=regionalize_foregrounds,
    accepted_locations=['RER', 'WEU'],
    locations_ranking=['WEU', 'CEU', 'RER', 'Europe without Switzerland', 'CH', 'DE', 'RoW', 'GLO'],
)

In [11]:
esm.change_location_mapping_file()

In [12]:
missing_flows = time_steps[0]['main_database'].test_mapping_file(esm.mapping[esm.mapping.Year == years[0]])

2026-11-06 17:53:24,729 - Database - INFO - Mapping successfully linked to the database


In [13]:
esm.check_inputs()

2026-11-06 17:53:24,851 - Mescal - WARNING - List of technologies or resources that are in the mapping file but not in the model file (this will not be a problem in the workflow): ['AC (Line)', 'H2 Store', 'H2 pipeline', 'battery', 'co2 stored (Store)', 'gas (Store)', 'gas pipeline', 'home battery (Store)', 'methanol (Store)', 'oil (Store)', 'rural water tanks (Store)', 'urban central water pits (Store)', 'urban central water tanks (Store)', 'urban decentral water tanks (Store)']
2026-11-06 17:53:24,926 - Mescal - WARNING - List of flows that are in the model file but not in the ESM flows to CPC mapping file. It might be an issue for double counting if these flows are inputs of some ESM technologies: ['solid biomass import']
2026-11-06 17:53:42,427 - Mescal - WARNING - Some technologies have no lifetime value for LCA in the lifetime file. Therefore, lifetime harmonization with the ESM will not be performed during the LCIA phase and capacity factor harmonization during the feedback of E

## Create the ESM databases

In [14]:
esm.create_esm_database()

2026-11-06 17:53:51,220 - Mescal - INFO - Starting to remove double-counted flows
100%|██████████| 57/57 [00:08<00:00,  6.78it/s]
2026-11-06 17:54:01,696 - Mescal - INFO - Double-counting removal done in 10.5 seconds
2026-11-06 17:54:01,814 - Mescal - WARNING - No construction input flow was found in the following operation datasets (this is not necessarily an issue, but you might want to have a look): ['urban central resistive heater']
2026-11-06 17:54:01,820 - Mescal - WARNING - Some input flows could not be found in the following operation datasets (this might be due to the operation dataset being a proxy of the technology or to an unmapped CPC category): {'DAC': ['co2'], 'HVC to air': ['non-sequestered HVC'], 'biogas to gas': ['co2'], 'process emissions (Link)': ['process emissions'], 'process emissions CC': ['process emissions', 'waste heat'], 'solid biomass for industry CC': ['co2'], 'urban central solid biomass CHP CC': ['co2'], 'land transport EV (Load)': ['EV battery']}
2026-1

Title: Writing activities to SQLite3 database:
  Started: 11/06/2026 17:55:16
  Finished: 11/06/2026 17:55:16
  Total time elapsed: 00:00:00
  CPU %: 107.80
  Memory %: 10.09


2026-11-06 17:57:44,855 - Database - INFO - PyPSA-Eur_2050 written to Brightway!
2026-11-06 17:57:47,098 - Mescal - INFO - Database written in 220.2 seconds


## Compute the impact scores

In [15]:
# Life-cycle impact scores
impact_scores, contrib_analysis, _ = esm.compute_impact_scores(
    methods=lcia_methods,
    contribution_analysis='both',
    contribution_analysis_limit_type='number',
    contribution_analysis_limit=10,
)
impact_scores.to_csv(f'{esm.results_path_file}/impact_scores.csv', index=False)
contrib_analysis.to_csv(f'{esm.results_path_file}/contribution_analysis.csv', index=False)

Getting activity data


100%|██████████| 149/149 [00:00<00:00, 74363.55it/s]


Adding exchange data to activities


100%|██████████| 3073/3073 [00:00<00:00, 37330.81it/s]


Filling out exchange data


100%|██████████| 149/149 [00:01<00:00, 147.60it/s]
2026-11-06 17:58:43,983 - Database - INFO - Loaded PyPSA-Eur_2050 from brightway!


... (2 outputs left out)

Adding exchange data to activities


100%|██████████| 3073/3073 [00:00<00:00, 33598.52it/s]


Filling out exchange data


100%|██████████| 149/149 [00:00<00:00, 256.15it/s]
2026-11-06 17:58:44,702 - Database - INFO - Loaded PyPSA-Eur_2050 from brightway!
148it [01:14,  1.98it/s]


In [16]:
# Direct impact scores
impact_scores_direct, _, _ = esm.compute_impact_scores(
    methods=lcia_methods,
    assessment_type='direct emissions',
)
impact_scores_direct.to_csv(f'{esm.results_path_file}/impact_scores_direct.csv', index=False)

Getting activity data


100%|██████████| 149/149 [00:00<00:00, 49642.65it/s]


Adding exchange data to activities


100%|██████████| 3073/3073 [00:00<00:00, 32483.91it/s]


Filling out exchange data


100%|██████████| 149/149 [00:00<00:00, 260.62it/s]
2026-11-06 18:00:05,249 - Database - INFO - Loaded PyPSA-Eur_2050 from brightway!


... (4 outputs left out)

Filling out exchange data


100%|██████████| 149/149 [00:00<00:00, 248.48it/s]
2026-11-06 18:00:05,957 - Database - INFO - Loaded PyPSA-Eur_2050 from brightway!
Writing activities to SQLite3 database:
0% [##############################] 100% | ETA: 00:00:00
Total time elapsed: 00:00:00


Title: Writing activities to SQLite3 database:
  Started: 11/06/2026 18:00:06
  Finished: 11/06/2026 18:00:07
  Total time elapsed: 00:00:00
  CPU %: 62.50
  Memory %: 9.99


2026-11-06 18:00:07,564 - Database - INFO - PyPSA-Eur_2050_direct_emissions written to Brightway!
57it [00:00, 224.13it/s]
